# Single meeting evaluation

Learn how individual DeepEval metrics relate to the complete evaluator under producer template **4.1** and methodology **2**. All fixtures are fictional. This notebook makes real, billable model requests. The optional teaching path repeats both metrics before running the full evaluator; disable it for routine work. No saved results are included.

## Setup

From the repository root install `.venv/bin/python -m pip install -e ".[dev,notebooks]"`, then start `.venv/bin/jupyter lab notebooks`. Select that environment as the kernel. Credentials belong in your local environment or `.env`, never notebook cells.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "pyproject.toml").exists())
load_dotenv(ROOT / ".env", override=False)
MEETING_ID = "meeting_001"
MODEL = "gpt-6-luna"
RUN_INDIVIDUAL_METRICS = True  # Extra teaching calls; use False for routine evaluation.
transcript = (ROOT / "examples/data/transcripts" / f"{MEETING_ID}.txt").read_text(encoding="utf-8")
summary = (ROOT / "examples/data/summaries" / f"{MEETING_ID}.md").read_text(encoding="utf-8")
provided_context = None  # Optional approved text received by the summary producer.
print("Loaded", MEETING_ID, len(transcript), "transcript characters")
print(summary)

## Evaluate with the current producer contract

The individual-metric test case uses the same packaged producer prompt and labeled source records as the evaluator. Supplied context is separate from the transcript and does not establish what was said at the meeting. `await` is required in Jupyter. This example owns the model clients explicitly and closes them even if a request fails.

In [ ]:
import json
import math
from deepeval.test_case import LLMTestCase
from langchain_openai import ChatOpenAI
from openai import DefaultAsyncHttpxClient, DefaultHttpxClient
from meeting_eval import MeetingSummaryEvaluator
from meeting_eval.metrics import create_metrics
from meeting_eval.diagnostics import BUSINESS_INSTRUCTIONS
from meeting_eval.policy import TEMPLATE_VERSION, METHODOLOGY_VERSION

llm = ChatOpenAI(
    model=MODEL, timeout=120, max_retries=2,
    http_client=DefaultHttpxClient(), http_async_client=DefaultAsyncHttpxClient(),
)
try:
    async with MeetingSummaryEvaluator(model=llm, max_concurrency=1, env_file=None) as evaluator:
        if RUN_INDIVIDUAL_METRICS:
            sources = [{"source": "transcript", "text": transcript}]
            if provided_context is not None:
                sources.append({"source": "context", "text": provided_context})
            test_case = LLMTestCase(
                input=BUSINESS_INSTRUCTIONS, actual_output=summary,
                retrieval_context=[json.dumps(source, ensure_ascii=False) for source in sources],
            )
            for name, metric in create_metrics(evaluator.judge).items():
                await metric.a_measure(test_case)
                assert metric.score is not None and math.isfinite(metric.score), name
                print("Individual", name, metric.score, metric.reason)
        result = await evaluator.aevaluate(transcript, summary, MEETING_ID, provided_context=provided_context)
finally:
    await llm.root_async_client.close()
    if llm.root_client is not None:
        llm.root_client.close()

assert result.execution_status == "completed", result.errors
assert result.metadata["diagnostics_status"] == "completed", result.errors
assert set(result.metrics) == {"faithfulness", "coverage"}
assert all(metric.status == "completed" and metric.score is not None and math.isfinite(metric.score)
           and 0 <= metric.score <= 1 for metric in result.metrics.values())
assert result.metadata["producer_template_version"] == TEMPLATE_VERSION
assert result.metadata["methodology_version"] == METHODOLOGY_VERSION
print("Execution:", result.execution_status, "Review required:", result.review_required)
for name, metric in result.metrics.items():
    print(name, metric.score, metric.reason)

## Inspect evidence and provenance

A verified quotation uniquely matches its indicated transcript, context, or summary source. That proves provenance, not that the judge interpreted it correctly. Missing or repeated quotations remain unverified. Review status is independent of execution: a completed evaluation can flag a material error.

In [ ]:
print("Review reasons:", result.review_reasons)
for finding in result.findings:
    print(f"[{finding.severity}] {finding.kind}: {finding.description}")
    for evidence in finding.evidence:
        print(evidence.source, evidence.line_start, evidence.line_end, "verified=", evidence.verified)
        print(evidence.quote)
    if finding.suggested_prompt_change:
        print("Prompt suggestion:", finding.suggested_prompt_change)
print("Run metadata:", result.metadata)
if not result.findings:
    print("No findings returned; this is not a guarantee of correctness.")

## Next experiments

Try `meeting_002` for central omissions and `meeting_003` for material distortions. To inspect supplied context, load `examples/data/contexts/meeting_001.md` into `provided_context` for the matching meeting. Generated outreach labeled `Suggested outreach:` may derive a relevant question from supported premises; `Discussed action:` requires explicit meeting support. Neither authorizes invented facts, products, endorsements, or deadlines.

Keep the judge, methodology, and input set fixed for producer-prompt comparisons. Inspect paired findings and repeat noisy cases. The calibration manifest contains draft boundary expectations, not human-reviewed accuracy labels.